In [1]:
import os
import pandas as pd


import torch
import torch.nn as nn
import torch.optim as optim


import torchvision
import torchvision.transforms as transforms

#from torchvision.models import densenet121
import timm
from torch.utils.data import DataLoader, random_split

print(timm.list_models('*vit*'))

['convit_base', 'convit_small', 'convit_tiny', 'crossvit_9_240', 'crossvit_9_dagger_240', 'crossvit_15_240', 'crossvit_15_dagger_240', 'crossvit_15_dagger_408', 'crossvit_18_240', 'crossvit_18_dagger_240', 'crossvit_18_dagger_408', 'crossvit_base_240', 'crossvit_small_240', 'crossvit_tiny_240', 'davit_base', 'davit_base_fl', 'davit_giant', 'davit_huge', 'davit_huge_fl', 'davit_large', 'davit_small', 'davit_tiny', 'efficientvit_b0', 'efficientvit_b1', 'efficientvit_b2', 'efficientvit_b3', 'efficientvit_l1', 'efficientvit_l2', 'efficientvit_l3', 'efficientvit_m0', 'efficientvit_m1', 'efficientvit_m2', 'efficientvit_m3', 'efficientvit_m4', 'efficientvit_m5', 'fastvit_ma36', 'fastvit_mci0', 'fastvit_mci1', 'fastvit_mci2', 'fastvit_mci3', 'fastvit_mci4', 'fastvit_s12', 'fastvit_sa12', 'fastvit_sa24', 'fastvit_sa36', 'fastvit_t8', 'fastvit_t12', 'flexivit_base', 'flexivit_large', 'flexivit_small', 'gcvit_base', 'gcvit_small', 'gcvit_tiny', 'gcvit_xtiny', 'gcvit_xxtiny', 'gemma4_vit_167m', 'g

In [ ]:
#KONIGURaCJa
# dataset:
# "CIFAR10"
# "CIFAR100"
DATASET_NAME = "CIFAR100"


BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-4


MODEL_SAVE_PATH_CHECKPOINT = f"models/t_checkpoint_smallvit_{DATASET_NAME.lower()}_{BATCH_SIZE}batch_{EPOCHS}epochs_{LEARNING_RATE}lr.pth"
MODEL_SAVE_PATH = f"models/smallvit_model_{DATASET_NAME.lower()}_{BATCH_SIZE}batch_{EPOCHS}epochs_{LEARNING_RATE}lr.pth"
RESULTS_TO_EXCEL = "training_results_VIT_CIFAR.xlsx"
# EARLY STOPPING
PATIENCE = 4

In [3]:
#TRANSFORMACJE
#vit wymaga wiekszych obrazow niz 32x32 wiec dodajemy resize do 224x224
IMAGE_SIZE=224

transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMAGE_SIZE, padding=4),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

transform_test = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])



In [4]:
#device
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("======================================")
print("DEVICE:", device)
print("======================================")

DEVICE: cuda


In [5]:
# ============================================================
# DATASET
# ============================================================

if DATASET_NAME == "CIFAR10":

    NUM_CLASSES = 10

    full_train_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=True,
        download=True,
        transform=transform_train
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=False,
        download=True,
        transform=transform_test
    )

elif DATASET_NAME == "CIFAR100":

    NUM_CLASSES = 100

    full_train_dataset = torchvision.datasets.CIFAR100(
        root="./data",
        train=True,
        download=True,
        transform=transform_train
    )

    test_dataset = torchvision.datasets.CIFAR100(
        root="./data",
        train=False,
        download=True,
        transform=transform_test
    )

else:
    raise ValueError("DATASET_NAME musi być CIFAR10 albo CIFAR100")

Files already downloaded and verified
Files already downloaded and verified


In [6]:
# ============================================================
# PODZIAŁ TRAIN / VALIDATION
# ============================================================

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

In [7]:
# DATALOADER
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

In [8]:
# MODEL DENSENET121


# # weights=None -> brak pretrained weights
# model = densenet121(weights=None)

# # zmiana pierwszej warstwy dla CIFAR
# model.features.conv0 = nn.Conv2d(
#     3, #licza kanałów wejściowych (RGB)
#     64, # liczba kanałów wyjściowych (zgodna z oryginalnym modelem)
#     kernel_size=3,# rozmiar filtra
#     stride=1, #krok przesuwania filtra
#     padding=1,#dodanie paddingu, żeby zachować rozmiar obrazu bo filtr 3x3 nie miesci sie
#     bias=False
# )

# # usunięcie maxpool
# model.features.pool0 = nn.Identity()

# # zmiana klasyfikatora
# model.classifier = nn.Sequential(
#     nn.Dropout(0.3),
#     nn.Linear(  
#     model.classifier.in_features,
#     NUM_CLASSES)
# )

# model = model.to(device)

#MODEL SMALLIT
model = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)
print("======================================")
print(model)
print("======================================")

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [9]:

# LOSS + OPTIMIZER


criterion = nn.CrossEntropyLoss() #funkcja bledu - porownuje przewidywania z prawdziwymi
# criterion = nn.CrossEntropyLoss(
#     label_smoothing=0.1
# )
# optimizer = optim.SGD(
#     model.parameters(),
#     lr=LEARNING_RATE,
#     momentum=0.9,
#     weight_decay=5e-4
# )
# optimizer = optim.Adam(
#     model.parameters(),
#     lr=LEARNING_RATE
# )
# optimizer = optim.SGD(
#     model.parameters(),
#     lr=LEARNING_RATE,
#     momentum=0.9,
#     weight_decay=5e-4
# )

# scheduler = optim.lr_scheduler.StepLR(
#     optimizer,
#     step_size=5,
#     gamma=0.1
# )

#OPTIMIZER DLA VIT zamiast sgd czy adam, bo vit lepiej sie uczy z adamw

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.05
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [10]:

# ============================================================
# EARLY STOPPING VARIABLES
# ============================================================

best_val_loss = float("inf")
early_stop_counter = 0


In [11]:
#pętla treningowa


print("======================================")
print("START TRENINGU")
print("======================================")

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        # zerowanie gradientów
        optimizer.zero_grad()

        # forward
        outputs = model(images)

        # loss
        loss = criterion(outputs, labels)

        # backward
        loss.backward() #liczenie gradientow

        # update wag
        optimizer.step()

        running_loss += loss.item()
        
        # accuracy train
        _, predicted = torch.max(outputs , 1)

        total_train += labels.size(0)

        correct_train += (predicted == labels).sum().item()
        
        # ====================================================
        # BATCH LOGGING
        # ====================================================

        if (batch_idx + 1) % 100 == 0:

            print(
                f"Epoch [{epoch+1}/{EPOCHS}] | "
                f"Batch [{batch_idx+1}/{len(train_loader)}] | "
                f"Batch Loss: {loss.item():.4f}"
            )

        
    # ========================================================
    # TRAIN METRICS
    # ========================================================

    train_loss = running_loss / len(train_loader)

    train_accuracy = 100 * correct_train / total_train
        
    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_loss = running_val_loss / len(val_loader)

    val_accuracy = 100 * correct / total

    # ========================================================
    # SCHEDULER STEP
    # ========================================================

    scheduler.step()
        

    # ========================================================
    # EPOCH LOGGING
    # ========================================================

    current_lr = optimizer.param_groups[0]["lr"]

    print("======================================")
    print(f"Epoch [{epoch+1}/{EPOCHS}] zakończona")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.2f}%")
    print(f"Val Loss       : {val_loss:.4f}")
    print(f"Val Accuracy   : {val_accuracy:.2f}%")
    print(f"Learning Rate  : {current_lr}")
    print("======================================")

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        early_stop_counter = 0

        torch.save(model.state_dict(), MODEL_SAVE_PATH)

        print("Najlepszy model zapisany.")

    else:

        early_stop_counter += 1

        print(f"Brak poprawy. Counter: {early_stop_counter}/{PATIENCE}")

    if early_stop_counter >= PATIENCE:

        print("======================================")
        print("EARLY STOPPING")
        print("======================================")
        break

print("======================================")
print("KONIEC TRENINGU")
print("======================================")

# ============================================================
# WCZYTANIE NAJLEPSZEGO MODELU
# ============================================================

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.to(device)




START TRENINGU
Epoch [1/10] | Batch [100/625] | Batch Loss: 0.9991
Epoch [1/10] | Batch [200/625] | Batch Loss: 0.6447
Epoch [1/10] | Batch [300/625] | Batch Loss: 0.9842
Epoch [1/10] | Batch [400/625] | Batch Loss: 0.6872
Epoch [1/10] | Batch [500/625] | Batch Loss: 0.5657
Epoch [1/10] | Batch [600/625] | Batch Loss: 0.4994
Epoch [1/10] zakończona
Train Loss     : 1.0078
Train Accuracy : 74.34%
Val Loss       : 0.5496
Val Accuracy   : 83.99%
Learning Rate  : 9.755282581475769e-05
Najlepszy model zapisany.
Epoch [2/10] | Batch [100/625] | Batch Loss: 0.4088
Epoch [2/10] | Batch [200/625] | Batch Loss: 0.5037
Epoch [2/10] | Batch [300/625] | Batch Loss: 0.2758
Epoch [2/10] | Batch [400/625] | Batch Loss: 0.4839
Epoch [2/10] | Batch [500/625] | Batch Loss: 0.3457
Epoch [2/10] | Batch [600/625] | Batch Loss: 0.5808
Epoch [2/10] zakończona
Train Loss     : 0.3623
Train Accuracy : 88.91%
Val Loss       : 0.5231
Val Accuracy   : 84.73%
Learning Rate  : 9.045084971874737e-05
Najlepszy model z

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [12]:
# EWALUACJA


model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("======================================")
print(f"Accuracy: {accuracy:.2f}%")
print("======================================")

Accuracy: 90.41%


In [13]:
# ============================================================
# ZAPIS WYNIKÓW DO EXCELA
# ============================================================

RESULTS_TO_EXCEL = "vit_training_results.xlsx"
OPTIMIZER_NAME = "AdamW"
SCHEDULER_NAME = "CosineAnnealingLR"
from datetime import datetime

new_result = {
    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Dataset": DATASET_NAME,
    "Model": "ViT Small Patch16 224",
    "Batch Size": BATCH_SIZE,
    "Max Epochs": EPOCHS,
    "Executed Epochs": epoch + 1,
    "Learning Rate": LEARNING_RATE,
    "Patience": PATIENCE,
    "Optimizer": OPTIMIZER_NAME,
    "Scheduler": SCHEDULER_NAME,

    "Train Accuracy": round(train_accuracy, 2),
    "Validation Accuracy": round(val_accuracy, 2),
    "Test Accuracy": round(accuracy, 2),
    "Best Validation Loss": round(best_val_loss, 4),
    "Model Path": MODEL_SAVE_PATH,
    "Device": str(device)
}

# ============================================================
# ODCZYT ISTNIEJĄCEGO PLIKU
# ============================================================

if os.path.exists(RESULTS_TO_EXCEL):

    try:

        df_existing = pd.read_excel(RESULTS_TO_EXCEL)

        df_new = pd.concat(
            [df_existing, pd.DataFrame([new_result])],
            ignore_index=True
        )

    except Exception as e:

        print("Błąd odczytu Excela:", e)

        print("Tworzenie nowego pliku Excel...")

        df_new = pd.DataFrame([new_result])

else:

    df_new = pd.DataFrame([new_result])

# ============================================================
# ZAPIS EXCEL
# ============================================================

try:

    df_new.to_excel(RESULTS_TO_EXCEL, index=False)

    print("======================================")
    print(f"Wyniki zapisane do: {RESULTS_TO_EXCEL}")
    print("======================================")

except Exception as e:

    print("======================================")
    print("BŁĄD ZAPISU EXCEL")
    print(e)
    print("======================================")

Wyniki zapisane do: vit_training_results.xlsx


In [14]:
# ============================================================
# ZAPIS MODELU
# ============================================================

checkpoint = {
    "epoch": epoch + 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_val_loss": best_val_loss,
    "dataset": DATASET_NAME,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
}

torch.save(checkpoint, MODEL_SAVE_PATH_CHECKPOINT)

print("======================================")
print(f"Model zapisany do: {MODEL_SAVE_PATH_CHECKPOINT}")
print("======================================")

Model zapisany do: models/t_checkpoint_cifar100_64batch_10epochs_0.0001lr.pth
